In [1]:
import rasterio
import matplotlib
import numpy as np
from sedona.raster_utils import SedonaUtils
import pyspark.sql.functions as f
from sedona.spark import SedonaContext
import os

/tmp/ipykernel_315/1102481671.py:4: DeprecationWarning: Importing from 'sedona.raster_utils.SedonaUtils' is deprecated. Please use 'sedona.spark.raster_utils.SedonaUtils' instead.
  from sedona.raster_utils import SedonaUtils


In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder().\
    config("spark.driver.memory", "2G")

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/07 17:55:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/07 17:55:37 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/07 17:55:37 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/07 17:55:37 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/07 17:55:37 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/12/07 17:55:37 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/07 17:55:37 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
geotiff_df = sedona\
    .read\
    .format("binaryFile")\
    .load(f"s3a://{bucket_name}/source_data/raster/japan_landsat_part.tif")

In [4]:
geotiff_df\
    .selectExpr("RS_FromGeoTiff(content) AS raster")\
    .createOrReplaceTempView("japan")

In [5]:
sedona\
    .sql("SELECT EXPLODE(RS_Tile(raster, 10, 10)) AS tile FROM japan")\
    .createOrReplaceTempView("tiles")

In [6]:
wkt = 'POLYGON ((747726.6770902255084366 3856119.2088216841220856, 741397.5797568620182574 3853730.8702053208835423, 734590.8147002257173881 3856238.6257525025866926, 729455.8866750439628959 3861373.5537776844576001, 732321.8930146803613752 3866030.8140795934014022, 735784.9840084075694904 3868657.9865575931034982, 733754.8961844984441996 3871046.3251739568077028, 729336.4697442258475348 3869374.4881425024941564, 723365.6232033168198541 3864836.6447714115493000, 718350.1121089532971382 3861731.8045701389200985, 713692.8518070442369208 3859343.4659537752158940, 708796.7576434988295659 3862925.9738783207722008, 705572.5005114079685882 3869971.5727965934202075, 707244.3375428625149652 3876778.3378532296046615, 710468.5946749533759430 3885854.0245954110287130, 716678.2750774987507612 3892899.6235136836767197, 723007.3724108622409403 3894571.4605451384559274, 731127.7237064985092729 3896362.7145074112340808, 739725.7427254074718803 3896243.2975765927694738, 747129.5924361345823854 3894690.8774759564548731, 752264.5204613163368776 3892421.9557904112152755, 753936.3574927708832547 3889914.2002432295121253, 755011.1098701345035806 3886451.1092495019547641, 757638.2823481344385073 3885256.9399413201026618, 759310.1193795889848843 3882510.3505325019359589, 759310.1193795889848843 3879644.3441928657703102, 758354.7839330435963348 3878330.7579538659192622, 757757.6992789526702836 3876539.5039915931411088, 757877.1162097707856447 3874628.8330985023640096, 757996.5331405890174210 3872837.5791362295858562, 758593.6177946799434721 3871165.7421047748066485, 756682.9469015890499577 3870449.2405198658816516, 756563.5299707708181813 3868299.7357651386409998, 755966.4453166800085455 3864597.8109097750857472, 755369.3606625890824944 3861731.8045701389200985, 755369.3606625890824944 3859701.7167462296783924, 751309.1850147709483281 3857791.0458531389012933, 747726.6770902255084366 3856119.2088216841220856))'

In [7]:
sedona.sql(
    f"""
    WITH algebra_result AS (
        SELECT 
            RS_MapAlgebra(tile, 'D', 'out = (rast[1] - rast[0]) / (rast[1] + rast[0]);') AS raster,
            ST_SetSRID(ST_GeomFromText('{wkt}'), 32653) AS geom
        FROM tiles
    ),
    clipped AS (
        SELECT RS_Clip(raster, 1, geom) AS raster
        FROM algebra_result
        WHERE RS_Intersects(geom, raster)
    )
    SELECT raster
    FROM clipped
    """
).createOrReplaceTempView("clipped")

In [8]:
sedona.sql(
    """
    SELECT * FROM clipped
    """
).show()

[Stage 3:>                                                          (0 + 1) / 1]

+--------------------+
|              raster|
+--------------------+
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
|GridCoverage2D["g...|
+--------------------+
only showing top 20 rows

